# Etapa 2B — Rastreabilidade persistente
Cria a tabela `medallion_dev_controle.execucoes_eventos` e o Volume `medallion_dev_controle.evidencias`.

O teste registra duas tentativas bem-sucedidas do mesmo lote e uma **falha deliberada**. O resultado esperado é SUCESSO, SUCESSO, FALHOU. Falhas de armazenamento não são ignoradas.

Guarde o valor de RUN_IDS_PARA_RELEITURA para o notebook 03. As evidências JSON já ficam no Volume; o output visual é apenas um resumo. A criação dos objetos também valida as permissões necessárias.

**Parâmetros:** execute apenas a primeira célula de código para exibir os widgets. Preencha os campos e depois execute as demais células. Use **Run all** quando os parâmetros já estiverem preenchidos.


In [0]:
# catalogo: workspace, o mesmo catálogo das etapas anteriores.
# batch_id: deixe vazio para gerar um novo lote de teste. Informe um valor apenas para rastrear um lote escolhido.
# O teste cria tentativas distintas: SUCESSO, SUCESSO e FALHOU deliberado.
# Guarde a lista RUN_IDS_PARA_RELEITURA desta nova execução para o notebook 03.
# Execute somente este CMD, preencha os widgets e então prossiga.
# 1. Exibir parâmetros — execute esta célula antes de preencher os campos.
dbutils.widgets.text("catalogo", spark.catalog.currentCatalog())
dbutils.widgets.text("batch_id", "")

print("Preencha os widgets acima e depois execute as próximas células.")


In [0]:
# CMD 2 — Localizar a pasta Git e importar o pacote. Nenhum preenchimento adicional.
from pathlib import Path
import sys
atual = Path.cwd().resolve()
raiz = next((p for p in [atual, *atual.parents]
             if (p / "src" / "medallion").is_dir() and (p / "pyproject.toml").is_file()), None)
if raiz is None:
    raise RuntimeError("Abra este notebook na pasta Git; se necessário, use %cd com a raiz do projeto.")
if str(raiz / "src") not in sys.path:
    sys.path.insert(0, str(raiz / "src"))
from medallion.config import PipelineConfig



In [0]:
# CMD 3 — Executar a prova e persistir eventos/arquivos. Esperado: RASTREABILIDADE: OK e hashes válidos nas três tentativas.
from medallion.observability.proof import executar_prova

params = {"ambiente": "dev", "catalogo": dbutils.widgets.get("catalogo").strip()}
if dbutils.widgets.get("batch_id").strip():
    params["batch_id"] = dbutils.widgets.get("batch_id").strip()
config = PipelineConfig(**params)
relatorio = executar_prova(spark, config, raiz)
for tentativa in relatorio:
    print(tentativa)



## Coleta desta revalidação

Após executar todos os CMDs, confira a mensagem final esperada. Se ocorrer erro, preserve o output e interrompa a sequência.

Exporte este notebook como **IPYNB com outputs**, abra o arquivo e confirme que os resultados estão presentes. Salve o arquivo antes de limpar outputs ou mudar a sessão. Faça commit/push na branch `chore/estrutura-inicial`, registre o SHA e envie o export para revisão antes de avançar ao próximo notebook.

Sugestão de commit: `docs: revalida 02_validar_rastreabilidade com instrucoes de parametros`.

O commit sozinho não comprova que os outputs foram incluídos. A cópia pública do export será revisada para remover e-mails e caminhos pessoais, com manifesto e hashes, preservando a coleta anterior. Esta rodada só será registrada como validada após revisão do novo export.
